# 🌍 AI Stick — NLLB-200 on Colab T4 GPU

**What this does:**
- Runs the full NLLB-200 translation backend on Colab's free T4 GPU
- Opens a public cloudflared tunnel
- Automatically updates the Cloudflare Worker to route Vercel traffic here

**Flow after running:**
```
Vercel → Cloudflare Worker (stable URL) → this Colab T4 → NLLB-200
```

### Quick start
1. **Runtime → Change runtime type → T4 GPU**
2. Fill in your tokens in **Cell 2**
3. **Runtime → Run all**
4. Leave **Cell 6** running — it keeps everything alive

> The Cloudflare Worker URL never changes so Vercel never needs updating.

In [ ]:
# ── Cell 1: Anti-idle JS (keeps Colab from disconnecting) ────────────────────
from IPython.display import display, Javascript
display(Javascript('''
function keepAlive() {
  var btn = document.querySelector('.mdl-dialog__actions button:first-child');
  if (btn) btn.click();
  var cb = document.querySelector('colab-connect-button');
  if (cb && cb.shadowRoot) { var rb = cb.shadowRoot.querySelector('#connect'); if (rb) rb.click(); }
}
setInterval(keepAlive, 55000);
console.log('[anti-idle] running');
'''))
print('✅ Anti-idle JS started — keep this browser tab open')

In [ ]:
# ── Cell 2: YOUR TOKENS — fill these in ──────────────────────────────────────
import os

# Get from: https://huggingface.co/settings/tokens
HF_TOKEN = 'YOUR_HF_TOKEN_HERE'

# Cloudflare Worker name (set this to update the Worker's HF_SPACE_URL var)
# Get your CF API token from: https://dash.cloudflare.com/profile/api-tokens
# Needs: Workers Scripts:Edit permission
CF_API_TOKEN    = 'YOUR_CF_API_TOKEN_HERE'
CF_ACCOUNT_ID   = 'keithtwesigye74'
CF_WORKER_NAME  = 'lunyoro-translator-api'

# The HF Space ID (for updating its COLAB_URL secret)
HF_SPACE_ID = 'keithtwesigye/runyoro-translator-api'

os.environ.update({
    'HF_TOKEN':       HF_TOKEN,
    'HF_USERNAME':    'keithtwesigye',
    'DISABLE_NLLB':   '0',
    'DISABLE_MARIAN': '1',
    'DISABLE_ONNX':   '1',
    'CORS_ORIGINS': (
        'https://frontend-six-phi-25.vercel.app,'
        'https://horizonx.kathay.tech,'
        'https://runyoro-rutooro-translator.vercel.app,'
        'https://lunyoro-translator-api.keithtwesigye74.workers.dev,'
        'http://localhost:3002'
    ),
    'PORT':         '8000',
    'HISTORY_FILE': '/content/history.json',
})

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch to T4!'
print(f'✅ Tokens set | GPU: {gpu}')

In [ ]:
# ── Cell 3: Install packages + cloudflared ────────────────────────────────────
import subprocess, sys
print('Installing packages...')
pkgs = [
    'fastapi==0.115.0', 'uvicorn==0.30.0', 'python-multipart==0.0.9',
    'transformers>=4.41.0,<4.52.0', 'sentencepiece>=0.2.0', 'sacremoses>=0.1.1',
    'sentence-transformers>=3.0.0,<4.0.0', 'rapidfuzz>=3.9.1', 'huggingface_hub>=0.23.0',
    'openai>=1.50.0', 'python-dotenv>=1.0.1', 'python-docx>=1.1.0',
    'pypdf2>=3.0.1', 'pdfplumber>=0.11.0', 'Pillow>=10.0.0',
    'opencv-python-headless>=4.9.0', 'easyocr', 'psutil', 'requests>=2.32.0',
    'pandas>=2.2.2', 'scikit-learn>=1.5.0', 'httpx>=0.27.0', 'sacrebleu>=2.4.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Installing cloudflared...')
subprocess.run(['wget', '-q', '-O', '/usr/local/bin/cloudflared',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
], check=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
print('✅ All done')

In [ ]:
# ── Cell 4: Clone repo + download NLLB models (~5 GB) ────────────────────────
import os, subprocess, sys

REPO = '/content/TRANSLATOR'
BACKEND = f'{REPO}/lunyoro-translator/backend'

if not os.path.exists(REPO):
    print('Cloning repo...')
    subprocess.run(['git','clone','--depth=1',
        'https://github.com/K227-arch/TRANSLATOR.git', REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'])

os.chdir(BACKEND)

print('Downloading NLLB models from HuggingFace (~5 GB, 3-5 min)...')
subprocess.run([sys.executable, 'download_models.py', '--nllb-only'], check=False)

# Translation index
idx = f'{BACKEND}/model/translation_index.pkl'
if not os.path.exists(idx):
    print('Downloading translation index...')
    try:
        from huggingface_hub import hf_hub_download
        hf_hub_download(repo_id='keithtwesigye/lunyoro-dataset',
            filename='translation_index.pkl', repo_type='dataset',
            local_dir=f'{BACKEND}/model')
        print('✅ Index downloaded')
    except Exception as e:
        print(f'Building index locally ({e})...')
        subprocess.run([sys.executable, 'build_index.py'])
else:
    print('✅ Index already present')

print('✅ Setup complete')

In [ ]:
# ── Cell 5: Start backend + tunnel + update Cloudflare Worker ────────────────
import subprocess, threading, time, re, urllib.request, requests as rq, os

BACKEND = '/content/TRANSLATOR/lunyoro-translator/backend'
os.chdir(BACKEND)

server_proc = tunnel_proc = None
tunnel_url  = None

# ── Start uvicorn ─────────────────────────────────────────────────────────────
server_proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=BACKEND
)
print('Starting server...')
for i in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=3)
        print(f'✅ Server ready ({(i+1)*2}s)')
        break
    except: print(f'  waiting... {(i+1)*2}s', end='\r')

# ── Open cloudflared tunnel ────────────────────────────────────────────────────
def _run_tunnel():
    global tunnel_url, tunnel_proc
    tunnel_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in tunnel_proc.stdout:
        m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
        if m:
            tunnel_url = m.group(0)
            break

print('Opening tunnel...')
threading.Thread(target=_run_tunnel, daemon=True).start()
for _ in range(20):
    time.sleep(1)
    if tunnel_url: break

if not tunnel_url:
    print('⚠ Tunnel URL not detected'); exit()

print(f'\n✅ Tunnel: {tunnel_url}')

# ── Update Cloudflare Worker HF_SPACE_URL var ──────────────────────────────────
# This makes the Worker proxy to this Colab instead of the HF Space
if CF_API_TOKEN and CF_API_TOKEN != 'YOUR_CF_API_TOKEN_HERE':
    cf_headers = {'Authorization': f'Bearer {CF_API_TOKEN}', 'Content-Type': 'application/json'}
    # Update the Worker's HF_SPACE_URL variable via Cloudflare API
    r = rq.put(
        f'https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}/workers/scripts/{CF_WORKER_NAME}/settings',
        headers=cf_headers,
        json={'bindings': [{'type': 'plain_text', 'name': 'HF_SPACE_URL', 'text': tunnel_url}]},
        timeout=15
    )
    if r.status_code in (200, 201):
        print(f'✅ Cloudflare Worker updated → {tunnel_url}')
        print(f'   Vercel workers.dev URL stays the same — traffic now hits this Colab T4')
    else:
        print(f'⚠ CF Worker update: {r.status_code} {r.text[:100]}')
        print(f'   Manually update HF_SPACE_URL in wrangler.jsonc to: {tunnel_url}')
else:
    print('ℹ  CF_API_TOKEN not set — update Worker manually:')
    print(f'   wrangler vars put HF_SPACE_URL "{tunnel_url}"')

# ── Also update HF Space secret so it can use this URL as fallback ────────────
r2 = rq.post(
    f'https://huggingface.co/api/spaces/{HF_SPACE_ID}/secrets',
    headers={'Authorization': f'Bearer {HF_TOKEN}', 'Content-Type': 'application/json'},
    json={'key': 'COLAB_URL', 'value': tunnel_url}, timeout=15
)
print(f'✅ HF Space COLAB_URL updated: {r2.status_code}')

print(f'''
{'='*60}
✅  NLLB is running on Colab T4 GPU!

   Cloudflare Worker (stable): 
   https://lunyoro-translator-api.keithtwesigye74.workers.dev

   Direct Colab tunnel:
   {tunnel_url}
{'='*60}
''')

In [ ]:
# ── Cell 6: Keep-alive loop — LEAVE THIS RUNNING ─────────────────────────────
# Pings every 3 min, restarts server/tunnel if they die,
# re-updates the Cloudflare Worker if the tunnel URL changes.

import time, urllib.request, json, subprocess, threading, re, requests as rq
from datetime import datetime

PING_INTERVAL = 180   # 3 min
STATUS_EVERY  = 600   # 10 min
last_status   = time.time()
ping_count    = 0
last_url      = tunnel_url

def _alive():
    try: urllib.request.urlopen('http://localhost:8000/ping', timeout=5); return True
    except: return False

def _restart_server():
    global server_proc
    print(f'[{datetime.now():%H:%M}] ⚠ Server down — restarting...')
    try: server_proc.kill()
    except: pass
    import os; os.chdir('/content/TRANSLATOR/lunyoro-translator/backend')
    server_proc = subprocess.Popen(
        ['python','-m','uvicorn','main:app','--host','0.0.0.0','--port','8000'],
        cwd='/content/TRANSLATOR/lunyoro-translator/backend'
    )
    time.sleep(20)

def _update_worker(new_url):
    """Push new tunnel URL to Cloudflare Worker and HF Space."""
    if CF_API_TOKEN and CF_API_TOKEN != 'YOUR_CF_API_TOKEN_HERE':
        cf_headers = {'Authorization': f'Bearer {CF_API_TOKEN}', 'Content-Type': 'application/json'}
        rq.put(
            f'https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT_ID}/workers/scripts/{CF_WORKER_NAME}/settings',
            headers=cf_headers,
            json={'bindings': [{'type': 'plain_text', 'name': 'HF_SPACE_URL', 'text': new_url}]},
            timeout=15
        )
    rq.post(
        f'https://huggingface.co/api/spaces/{HF_SPACE_ID}/secrets',
        headers={'Authorization': f'Bearer {HF_TOKEN}', 'Content-Type': 'application/json'},
        json={'key': 'COLAB_URL', 'value': new_url}, timeout=15
    )

def _restart_tunnel():
    global tunnel_url, tunnel_proc, last_url
    print(f'[{datetime.now():%H:%M}] 🔁 Tunnel died — restarting...')
    try: tunnel_proc.kill()
    except: pass
    threading.Thread(target=_run_tunnel, daemon=True).start()
    for _ in range(20):
        time.sleep(1)
        if tunnel_url and tunnel_url != last_url: break
    if tunnel_url and tunnel_url != last_url:
        last_url = tunnel_url
        print(f'   New tunnel: {tunnel_url}')
        _update_worker(tunnel_url)
        print(f'   Worker + HF Space updated')

print(f'🔄 Keep-alive running — pinging every {PING_INTERVAL//60} min')
print(f'   Stop this cell to shut down everything.\n')

while True:
    time.sleep(PING_INTERVAL)
    ping_count += 1
    now = f'{datetime.now():%H:%M:%S}'

    if not _alive(): _restart_server()
    try: urllib.request.urlopen('http://localhost:8000/ping', timeout=5)
    except: pass

    if tunnel_proc and tunnel_proc.poll() is not None:
        _restart_tunnel()

    if time.time() - last_status >= STATUS_EVERY:
        last_status = time.time()
        try:
            with urllib.request.urlopen('http://localhost:8000/system-info', timeout=5) as r:
                info = json.loads(r.read())
            print(f'[{now}] NLLB:{info["nllb_en2lun"]} GPU:{info["gpu_available"]} pings:{ping_count}  {tunnel_url}')
        except Exception as e:
            print(f'[{now}] ⚠ {e}')

## After running

The Cloudflare Worker URL stays the same forever:
```
https://lunyoro-translator-api.keithtwesigye74.workers.dev
```
Set this as `NEXT_PUBLIC_API_URL` in Vercel once — never change it again.

When Colab is running → Worker routes to **Colab T4 (NLLB)**  
When Colab is off → Worker routes to **HF Space (Marian fallback)**

### Getting your Cloudflare API token
1. Go to https://dash.cloudflare.com/profile/api-tokens
2. Create token → use template **"Edit Cloudflare Workers"**
3. Paste it as `CF_API_TOKEN` in Cell 2